# Projet 5 - Machine Learning pour l'Allocation de Portefeuille

**Cours :** AI In Finance  
**Encadrants :** Nicolas De Roux & Mohamed El Fakir  
**Groupe :** Groupe 1  
**Membres :**  
- Maël Pertuisot  
- Valentin Martel  
- Mathias Garnier  

**Date :** 20/04/2026

---

## Objectif du projet

Construire un modèle d'allocation de portefeuille basé sur le Machine Learning.  
On sélectionne **50 actions du NASDAQ**, on calcule des features (rendements, volatilité,  
corrélations, secteurs), on prédit les rendements et la volatilité future, puis on  
construit un portefeuille optimisé (Markowitz / maximum Sharpe).  
On évalue la performance avec le **ratio de Sharpe**, le **drawdown maximum** et le **turnover**.

**Dataset de référence :** Stock Market Dataset (NASDAQ Universe)  
On utilise `yfinance` pour télécharger les données.

---

## 1. Installation et Imports

In [ ]:
!pip install yfinance scikit-learn torch matplotlib seaborn scipy --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

from scipy.optimize import minimize

import torch
import torch.nn as nn
import torch.optim as optim

pd.set_option('display.width', 130)
pd.set_option('display.max_columns', 20)

print("Imports OK")
print("Pandas:", pd.__version__)
print("PyTorch:", torch.__version__)

## 2. Collecte des données - 50 tickers NASDAQ

On sélectionne 50 actions du NASDAQ couvrant plusieurs secteurs.  
Période : 2018-01-01 à 2024-12-31.

In [ ]:
# 50 tickers NASDAQ répartis par secteur
tickers_par_secteur = {
    'Tech':        ['AAPL', 'MSFT', 'GOOGL', 'META', 'NVDA', 'ADBE', 'CRM', 'INTC', 'AMD', 'CSCO',
                    'AVGO', 'TXN', 'QCOM', 'AMAT', 'MU'],
    'Sante':       ['AMGN', 'GILD', 'MRNA', 'REGN', 'VRTX', 'BIIB', 'ILMN', 'DXCM'],
    'Conso':       ['AMZN', 'TSLA', 'COST', 'PEP', 'SBUX', 'MDLZ', 'KHC', 'MNST'],
    'Finance':     ['PYPL', 'ISRG', 'MELI', 'FTNT', 'NFLX'],
    'Industrie':   ['HON', 'CSX', 'ODFL', 'FAST', 'PAYX'],
    'Telecom':     ['CMCSA', 'TMUS', 'CHTR'],
    'Utilities':   ['XEL', 'AEP', 'EXC'],
    'Energie':     ['CEG', 'FANG', 'DDOG']
}

# Aplatir la liste
tickers = []
secteur_map = {}  # ticker -> secteur
for secteur, liste in tickers_par_secteur.items():
    for t in liste:
        tickers.append(t)
        secteur_map[t] = secteur

print(f"Nombre de tickers : {len(tickers)}")
print(f"Secteurs : {list(tickers_par_secteur.keys())}")

In [ ]:
# Téléchargement des données
data_brut = yf.download(
    tickers=tickers,
    start='2018-01-01',
    end='2025-01-01',
    auto_adjust=False
)

print("Shape données brutes:", data_brut.shape)
data_brut.head()

In [ ]:
# Extraire les prix de clôture ajustés
prices = data_brut['Adj Close'].copy()

# Enlever les tickers avec trop de NaN (pas cotés sur toute la période)
pct_nan = prices.isna().mean()
tickers_valides = pct_nan[pct_nan < 0.05].index.tolist()
prices = prices[tickers_valides].dropna()

# Mettre à jour la liste de tickers
tickers = tickers_valides
print(f"Tickers conservés : {len(tickers)} / 50")
print(f"Période : {prices.index[0].date()} à {prices.index[-1].date()}")
print(f"Nombre de jours : {len(prices)}")

In [ ]:
# Extraire aussi les volumes
volumes = data_brut['Volume'][tickers].copy()
volumes = volumes.loc[prices.index]

# Créer un DataFrame des secteurs
df_secteurs = pd.DataFrame({
    'ticker': tickers,
    'secteur': [secteur_map.get(t, 'Autre') for t in tickers]
}).set_index('ticker')

print("\nRépartition par secteur :")
print(df_secteurs['secteur'].value_counts())

## 3. Exploration des données (EDA)

In [ ]:
# Rendements journaliers
returns = prices.pct_change().dropna()

print("Shape des rendements:", returns.shape)
returns.describe().T.head(10)

In [ ]:
# Statistiques par secteur
stats_secteur = pd.DataFrame({
    'Rendement moyen (ann.)': returns.mean() * 252,
    'Volatilité (ann.)': returns.std() * np.sqrt(252),
    'Secteur': [secteur_map.get(t, 'Autre') for t in returns.columns]
})

print("\nStatistiques annualisées par secteur :")
print(stats_secteur.groupby('Secteur').mean().round(4))

In [ ]:
# Evolution de quelques prix normalisés (base 100), un par secteur
exemples = ['AAPL', 'AMGN', 'AMZN', 'PYPL', 'HON', 'CMCSA', 'XEL']
exemples = [t for t in exemples if t in tickers]

prix_norm = prices[exemples] / prices[exemples].iloc[0] * 100

plt.figure(figsize=(12, 6))
for col in prix_norm.columns:
    plt.plot(prix_norm.index, prix_norm[col], label=f"{col} ({secteur_map.get(col, '')})")
plt.title('Prix normalisés (base 100) - 1 action par secteur')
plt.xlabel('Date')
plt.ylabel('Prix normalisé')
plt.legend(fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Matrice de corrélation moyenne par secteur
returns_secteur = pd.DataFrame()
for secteur in tickers_par_secteur.keys():
    cols = [t for t in tickers_par_secteur[secteur] if t in tickers]
    if len(cols) > 0:
        returns_secteur[secteur] = returns[cols].mean(axis=1)

plt.figure(figsize=(8, 6))
sns.heatmap(returns_secteur.corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Corrélation entre secteurs (rendements moyens)')
plt.tight_layout()
plt.show()

In [ ]:
# Volatilité glissante moyenne par secteur
vol_30j = returns.rolling(30).std() * np.sqrt(252)

vol_secteur = pd.DataFrame()
for secteur in tickers_par_secteur.keys():
    cols = [t for t in tickers_par_secteur[secteur] if t in tickers]
    if len(cols) > 0:
        vol_secteur[secteur] = vol_30j[cols].mean(axis=1)

plt.figure(figsize=(12, 5))
for col in vol_secteur.columns:
    plt.plot(vol_secteur.index, vol_secteur[col], label=col, alpha=0.7)
plt.title('Volatilité annualisée glissante (30j) par secteur')
plt.xlabel('Date')
plt.ylabel('Volatilité')
plt.legend(fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Feature Engineering

Pour chaque action on crée des features :
- Rendements passés (lags 1 à 5)
- Moyennes mobiles (5j, 20j) et leur ratio
- Volatilité glissante (10j et 20j)
- RSI simplifié (14j)
- Momentum 5j et 20j
- Volume relatif
- Corrélation glissante avec le marché
- Secteur (one-hot encoding)

**Target : rendement de la semaine suivante** (next-week return, 5 jours)

In [ ]:
# Rendement moyen du marché (pour la corrélation)
rendement_marche = returns.mean(axis=1)

def creer_features_action(prix_serie, ret_serie, vol_serie, rendement_mkt, ticker):
    """
    Crée les features pour une action.
    Target = rendement cumulé sur les 5 prochains jours.
    """
    df = pd.DataFrame(index=prix_serie.index)
    
    # Rendement journalier
    df['ret'] = ret_serie
    
    # Lags des rendements
    for lag in range(1, 6):
        df[f'ret_lag{lag}'] = df['ret'].shift(lag)
    
    # Moyennes mobiles
    df['ma5'] = prix_serie.rolling(5).mean()
    df['ma20'] = prix_serie.rolling(20).mean()
    df['ma_ratio'] = df['ma5'] / df['ma20']
    
    # Volatilité glissante
    df['vol_10j'] = df['ret'].rolling(10).std()
    df['vol_20j'] = df['ret'].rolling(20).std()
    
    # RSI (14 jours)
    delta = prix_serie.diff()
    gain = delta.where(delta > 0, 0).rolling(14).mean()
    perte = (-delta.where(delta < 0, 0)).rolling(14).mean()
    rs = gain / perte
    df['rsi'] = 100 - (100 / (1 + rs))
    
    # Momentum
    df['momentum_5j'] = prix_serie.pct_change(5)
    df['momentum_20j'] = prix_serie.pct_change(20)
    
    # Volume relatif (si disponible)
    if vol_serie is not None:
        df['vol_rel'] = vol_serie / vol_serie.rolling(20).mean()
    
    # Corrélation glissante avec le marché (30j)
    df['corr_marche'] = df['ret'].rolling(30).corr(rendement_mkt)
    
    # Target : rendement cumulé sur les 5 prochains jours
    df['target'] = prix_serie.pct_change(5).shift(-5)
    
    # On supprime les colonnes intermédiaires
    df = df.drop(columns=['ret', 'ma5', 'ma20'])
    
    df = df.dropna()
    return df

In [ ]:
# Créer les features pour toutes les actions
features_dict = {}
for ticker in tickers:
    features_dict[ticker] = creer_features_action(
        prices[ticker], returns[ticker],
        volumes[ticker] if ticker in volumes.columns else None,
        rendement_marche, ticker
    )

print(f"Features créées pour {len(features_dict)} actions")
exemple = list(features_dict.keys())[0]
print(f"Exemple ({exemple}) : {features_dict[exemple].shape[0]} obs, {features_dict[exemple].shape[1]-1} features")
print(f"Colonnes : {features_dict[exemple].columns.tolist()}")

In [ ]:
# Empiler toutes les actions + one-hot encoding du secteur
liste_dfs = []
for ticker in tickers:
    df_tmp = features_dict[ticker].copy()
    df_tmp['ticker'] = ticker
    df_tmp['secteur'] = secteur_map.get(ticker, 'Autre')
    liste_dfs.append(df_tmp)

df_all = pd.concat(liste_dfs)
df_all = df_all.reset_index().rename(columns={'index': 'Date'})

# One-hot encoding du secteur
df_all = pd.get_dummies(df_all, columns=['secteur'], prefix='sect')

print("Shape totale:", df_all.shape)
df_all.head()

## 5. Modélisation - Prédiction des rendements hebdomadaires

On entraîne les modèles sur toutes les actions empilées (panel).  
Split temporel : train avant 2023, test à partir de 2023.  
3 modèles : Régression Linéaire, Ridge, Random Forest.

In [ ]:
# Séparer les colonnes
cols_features = [c for c in df_all.columns if c not in ['Date', 'ticker', 'target']]
cols_secteur = [c for c in cols_features if c.startswith('sect_')]

# Split temporel
date_split = '2023-01-01'
mask_train = df_all['Date'] < date_split
mask_test = df_all['Date'] >= date_split

X_train = df_all.loc[mask_train, cols_features].values
y_train = df_all.loc[mask_train, 'target'].values
X_test = df_all.loc[mask_test, cols_features].values
y_test = df_all.loc[mask_test, 'target'].values

# Info test pour reconstruire les prédictions par ticker/date
info_test = df_all.loc[mask_test, ['Date', 'ticker']].copy()

print(f"Train : {mask_train.sum()} observations")
print(f"Test  : {mask_test.sum()} observations")
print(f"Features : {len(cols_features)}")

In [ ]:
# Scaling (on ne scale pas les colonnes one-hot)
cols_numeriques = [c for c in cols_features if c not in cols_secteur]
idx_num = [cols_features.index(c) for c in cols_numeriques]

scaler = StandardScaler()
X_train_sc = X_train.copy().astype(float)
X_test_sc = X_test.copy().astype(float)

X_train_sc[:, idx_num] = scaler.fit_transform(X_train[:, idx_num])
X_test_sc[:, idx_num] = scaler.transform(X_test[:, idx_num])

print("Scaling OK")

In [ ]:
# Modèle 1 : Régression Linéaire
lr = LinearRegression()
lr.fit(X_train_sc, y_train)
pred_lr = lr.predict(X_test_sc)

# Modèle 2 : Ridge
ridge = Ridge(alpha=10.0)
ridge.fit(X_train_sc, y_train)
pred_ridge = ridge.predict(X_test_sc)

# Modèle 3 : Random Forest
rf = RandomForestRegressor(n_estimators=100, max_depth=6, 
                           min_samples_leaf=20, random_state=42)
rf.fit(X_train_sc, y_train)
pred_rf = rf.predict(X_test_sc)

print("Modèles entraînés")

In [ ]:
# Evaluation des modèles
resultats = {}
for nom, pred in [('LinearReg', pred_lr), ('Ridge', pred_ridge), ('RandomForest', pred_rf)]:
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    mae = mean_absolute_error(y_test, pred)
    r2 = r2_score(y_test, pred)
    resultats[nom] = {'RMSE': rmse, 'MAE': mae, 'R2': r2}

resultats_df = pd.DataFrame(resultats).T
print("\n=== Métriques de prédiction (next-week return) ===")
print(resultats_df.round(6))

In [ ]:
# Feature importance du Random Forest
importance = pd.DataFrame({
    'feature': cols_features,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
top_n = 15
plt.barh(importance['feature'].iloc[:top_n], importance['importance'].iloc[:top_n])
plt.xlabel('Importance')
plt.title(f'Top {top_n} features - Random Forest')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 6. Modèle LSTM

On entraîne un LSTM sur un sous-ensemble de 10 tickers pour comparer.  
Fenêtre de 10 jours pour prédire le rendement hebdo suivant.

In [ ]:
# Préparer les séquences pour le LSTM
window = 10

def create_sequences(X, y, window_size):
    Xs, ys = [], []
    for i in range(len(X) - window_size):
        Xs.append(X[i:i+window_size])
        ys.append(y[i+window_size])
    return np.array(Xs), np.array(ys)

# Sous-ensemble pour le LSTM (sinon trop long)
tickers_lstm = tickers

X_seq_train_list, y_seq_train_list = [], []
X_seq_test_list, y_seq_test_list = [], []

for ticker in tickers_lstm:
    df_t = df_all[df_all['ticker'] == ticker].sort_values('Date')
    
    X_t = df_t[cols_features].values.astype(float)
    y_t = df_t['target'].values.astype(float)
    
    # Scaling par ticker
    sc = StandardScaler()
    mask_tr = df_t['Date'] < date_split
    X_t_sc = X_t.copy()
    X_t_sc[mask_tr.values, :len(cols_numeriques)] = sc.fit_transform(X_t[mask_tr.values, :len(cols_numeriques)])
    X_t_sc[~mask_tr.values, :len(cols_numeriques)] = sc.transform(X_t[~mask_tr.values, :len(cols_numeriques)])
    
    n_train = mask_tr.sum()
    
    X_tr, y_tr = create_sequences(X_t_sc[:n_train], y_t[:n_train], window)
    X_te, y_te = create_sequences(X_t_sc[n_train:], y_t[n_train:], window)
    
    if len(X_tr) > 0:
        X_seq_train_list.append(X_tr)
        y_seq_train_list.append(y_tr)
    if len(X_te) > 0:
        X_seq_test_list.append(X_te)
        y_seq_test_list.append(y_te)

X_seq_train = np.concatenate(X_seq_train_list)
y_seq_train = np.concatenate(y_seq_train_list)
X_seq_test = np.concatenate(X_seq_test_list)
y_seq_test = np.concatenate(y_seq_test_list)

print(f"LSTM - Train: {X_seq_train.shape}, Test: {X_seq_test.shape}")

In [ ]:
# Conversion en tensors
X_train_t = torch.tensor(X_seq_train, dtype=torch.float32)
y_train_t = torch.tensor(y_seq_train, dtype=torch.float32).unsqueeze(1)
X_test_t = torch.tensor(X_seq_test, dtype=torch.float32)
y_test_t = torch.tensor(y_seq_test, dtype=torch.float32).unsqueeze(1)

# Définition du LSTM
class LSTMRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dim=32):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
    
    def forward(self, x):
        lstm_out, (hn, cn) = self.lstm(x)
        out = self.fc(hn[-1])
        return out

input_dim = X_train_t.shape[2]
model_lstm = LSTMRegressor(input_dim=input_dim, hidden_dim=32)
criterion = nn.MSELoss()
optimizer_lstm = optim.Adam(model_lstm.parameters(), lr=1e-3)

print(model_lstm)

In [ ]:
# Entraînement
epochs = 30
batch_size = 128
train_losses = []
val_losses = []

for epoch in range(epochs):
    model_lstm.train()
    idx = torch.randperm(X_train_t.size(0))
    X_shuf = X_train_t[idx]
    y_shuf = y_train_t[idx]
    
    epoch_loss = 0
    n_batch = 0
    for i in range(0, X_train_t.size(0), batch_size):
        xb = X_shuf[i:i+batch_size]
        yb = y_shuf[i:i+batch_size]
        
        preds = model_lstm(xb)
        loss = criterion(preds, yb)
        
        optimizer_lstm.zero_grad()
        loss.backward()
        optimizer_lstm.step()
        
        epoch_loss += loss.item()
        n_batch += 1
    
    train_losses.append(epoch_loss / n_batch)
    
    model_lstm.eval()
    with torch.no_grad():
        val_pred = model_lstm(X_test_t)
        val_loss = criterion(val_pred, y_test_t)
        val_losses.append(val_loss.item())
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs} - Train: {train_losses[-1]:.6f} - Val: {val_losses[-1]:.6f}")

# Courbe de loss
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label='Train')
plt.plot(val_losses, label='Validation')
plt.title('Courbe d\'apprentissage LSTM')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluation LSTM
model_lstm.eval()
with torch.no_grad():
    pred_lstm_test = model_lstm(X_test_t).numpy().flatten()

rmse_lstm = np.sqrt(mean_squared_error(y_seq_test, pred_lstm_test))
r2_lstm = r2_score(y_seq_test, pred_lstm_test)

print(f"LSTM - RMSE: {rmse_lstm:.6f}, R2: {r2_lstm:.6f}")
print(f"\nComparaison des RMSE :")
print(f"  Linear   : {resultats_df.loc['LinearReg', 'RMSE']:.6f}")
print(f"  Ridge    : {resultats_df.loc['Ridge', 'RMSE']:.6f}")
print(f"  RF       : {resultats_df.loc['RandomForest', 'RMSE']:.6f}")
print(f"  LSTM     : {rmse_lstm:.6f}  (sur 10 tickers seulement)")

## 7. Construction du portefeuille

On utilise les prédictions du **Random Forest** (sur les 50 tickers)  
pour construire un portefeuille rééquilibré chaque semaine.

### Trois stratégies comparées :
1. **Equal Weight (1/N)** - benchmark, poids identiques
2. **ML Long-Only** - allocation proportionnelle aux rendements prédits positifs
3. **ML + Markowitz** - optimisation Mean-Variance avec prédictions ML et matrice de covariance historique

In [ ]:
# Reconstruire les prédictions par (date, ticker) sur la période test
info_test = info_test.copy()
info_test['pred_rf'] = pred_rf
info_test['target'] = y_test

# Pivoter : lignes = dates, colonnes = tickers
pred_pivot = info_test.pivot(index='Date', columns='ticker', values='pred_rf')
target_pivot = info_test.pivot(index='Date', columns='ticker', values='target')

# Garder les colonnes/dates complètes
pred_pivot = pred_pivot.dropna(axis=1, how='all').dropna(axis=0)
target_pivot = target_pivot.loc[pred_pivot.index, pred_pivot.columns]

# Rééquilibrage hebdomadaire (toutes les 5 séances)
dates_hebdo = pred_pivot.index[::5]
pred_hebdo = pred_pivot.loc[dates_hebdo]
target_hebdo = target_pivot.loc[dates_hebdo]

tickers_portf = pred_hebdo.columns.tolist()
n_assets = len(tickers_portf)

print(f"Dates de rééquilibrage : {len(dates_hebdo)}")
print(f"Actifs dans le portefeuille : {n_assets}")

In [ ]:
# --- Stratégie 1 : Equal Weight ---
poids_ew = np.ones(n_assets) / n_assets
rendement_ew = (target_hebdo * poids_ew).sum(axis=1)

# --- Stratégie 2 : ML Long-Only ---
def poids_ml_longonly(preds_row):
    p = preds_row.clip(lower=0)
    s = p.sum()
    if s > 0:
        return p / s
    else:
        return pd.Series(1.0/len(preds_row), index=preds_row.index)

poids_ml = pred_hebdo.apply(poids_ml_longonly, axis=1)
rendement_ml = (poids_ml * target_hebdo).sum(axis=1)

In [ ]:
# --- Stratégie 3 : ML + Markowitz (Maximum Sharpe) ---

returns_test = returns.loc[returns.index >= date_split, tickers_portf]

def optimiser_markowitz(mu_pred, cov_matrix):
    """
    Maximiser le ratio de Sharpe.
    Contrainte : poids >= 0, somme = 1, max 10% par action.
    """
    n = len(mu_pred)
    
    def neg_sharpe(w):
        ret_p = np.dot(w, mu_pred)
        vol_p = np.sqrt(np.dot(w, np.dot(cov_matrix, w)))
        if vol_p < 1e-10:
            return 0
        return -ret_p / vol_p
    
    contraintes = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
    bornes = [(0, 0.10)] * n  # max 10% par action
    w0 = np.ones(n) / n
    
    result = minimize(neg_sharpe, w0, method='SLSQP',
                      bounds=bornes, constraints=contraintes,
                      options={'maxiter': 200})
    
    if result.success:
        return result.x
    else:
        return np.ones(n) / n  # fallback EW

# Calculer les poids pour chaque date de rééquilibrage
poids_mkw = pd.DataFrame(index=dates_hebdo, columns=tickers_portf, dtype=float)

for date in dates_hebdo:
    mu = pred_hebdo.loc[date].values
    
    # Covariance historique (60 derniers jours)
    date_idx = returns_test.index.get_indexer([date], method='ffill')[0]
    debut = max(0, date_idx - 60)
    cov = returns_test.iloc[debut:date_idx].cov().values
    
    if date_idx - debut < 20:
        poids_mkw.loc[date] = 1.0 / n_assets
    else:
        w = optimiser_markowitz(mu, cov)
        poids_mkw.loc[date] = w

rendement_mkw = (poids_mkw.astype(float) * target_hebdo).sum(axis=1)

print("Portefeuilles construits")

## 8. Évaluation des portefeuilles

In [ ]:
# Fonctions d'évaluation

def sharpe_ratio(rendements, freq=52):
    """Ratio de Sharpe annualisé."""
    if rendements.std() == 0:
        return 0
    return np.sqrt(freq) * rendements.mean() / rendements.std()

def max_drawdown(rendements):
    """Drawdown maximum."""
    cumul = (1 + rendements).cumprod()
    pic = cumul.cummax()
    dd = (cumul - pic) / pic
    return dd.min()

def rendement_annualise(rendements, freq=52):
    """Rendement annualisé."""
    cumul = (1 + rendements).prod()
    n_ans = len(rendements) / freq
    if n_ans <= 0:
        return 0
    return cumul ** (1 / n_ans) - 1

def turnover(poids_df):
    """Turnover moyen hebdomadaire."""
    diff_poids = poids_df.diff().abs().sum(axis=1)
    return diff_poids.mean()

In [ ]:
# Poids EW (constant) pour le calcul du turnover
poids_ew_df = pd.DataFrame(
    np.ones((len(dates_hebdo), n_assets)) / n_assets,
    index=dates_hebdo, columns=tickers_portf
)

# Tableau de métriques
metriques = pd.DataFrame({
    'Equal Weight': [
        rendement_annualise(rendement_ew),
        rendement_ew.std() * np.sqrt(52),
        sharpe_ratio(rendement_ew),
        max_drawdown(rendement_ew),
        turnover(poids_ew_df)
    ],
    'ML Long-Only': [
        rendement_annualise(rendement_ml),
        rendement_ml.std() * np.sqrt(52),
        sharpe_ratio(rendement_ml),
        max_drawdown(rendement_ml),
        turnover(poids_ml)
    ],
    'ML + Markowitz': [
        rendement_annualise(rendement_mkw),
        rendement_mkw.std() * np.sqrt(52),
        sharpe_ratio(rendement_mkw),
        max_drawdown(rendement_mkw),
        turnover(poids_mkw)
    ]
}, index=['Rendement annualisé', 'Volatilité annualisée', 'Ratio de Sharpe',
          'Max Drawdown', 'Turnover moyen'])

print("\n=== Comparaison des portefeuilles ===")
print(metriques.round(4))

In [ ]:
# Rendement cumulé
cumul_ew = (1 + rendement_ew).cumprod()
cumul_ml = (1 + rendement_ml).cumprod()
cumul_mkw = (1 + rendement_mkw).cumprod()

plt.figure(figsize=(12, 6))
plt.plot(cumul_ew.index, cumul_ew.values, label='Equal Weight (1/N)', linewidth=2, linestyle='--')
plt.plot(cumul_ml.index, cumul_ml.values, label='ML Long-Only', linewidth=2)
plt.plot(cumul_mkw.index, cumul_mkw.values, label='ML + Markowitz', linewidth=2)
plt.title('Rendement cumulé des 3 stratégies')
plt.xlabel('Date')
plt.ylabel('Valeur (base 1)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Drawdown
def serie_drawdown(rendements):
    cumul = (1 + rendements).cumprod()
    return (cumul - cumul.cummax()) / cumul.cummax()

dd_ew = serie_drawdown(rendement_ew)
dd_ml = serie_drawdown(rendement_ml)
dd_mkw = serie_drawdown(rendement_mkw)

plt.figure(figsize=(12, 4))
plt.fill_between(dd_ew.index, dd_ew.values, 0, alpha=0.3, label='Equal Weight')
plt.fill_between(dd_ml.index, dd_ml.values, 0, alpha=0.3, label='ML Long-Only')
plt.fill_between(dd_mkw.index, dd_mkw.values, 0, alpha=0.3, label='ML + Markowitz')
plt.title('Drawdown des 3 stratégies')
plt.ylabel('Drawdown')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Allocation moyenne par secteur (Markowitz)
alloc_secteur = pd.DataFrame(index=poids_mkw.index)
for secteur in tickers_par_secteur.keys():
    cols = [t for t in tickers_par_secteur[secteur] if t in tickers_portf]
    if len(cols) > 0:
        alloc_secteur[secteur] = poids_mkw[cols].astype(float).sum(axis=1)

alloc_moy = alloc_secteur.mean().sort_values(ascending=False)

plt.figure(figsize=(8, 5))
plt.bar(alloc_moy.index, alloc_moy.values)
plt.title('Allocation moyenne par secteur - ML + Markowitz')
plt.ylabel('Poids moyen')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("\nAllocation moyenne par secteur :")
print(alloc_moy.round(4))

In [ ]:
# Turnover au cours du temps
turnover_ml_ts = poids_ml.diff().abs().sum(axis=1)
turnover_mkw_ts = poids_mkw.astype(float).diff().abs().sum(axis=1)

plt.figure(figsize=(12, 4))
plt.plot(turnover_ml_ts.index[1:], turnover_ml_ts.values[1:], label='ML Long-Only', alpha=0.7)
plt.plot(turnover_mkw_ts.index[1:], turnover_mkw_ts.values[1:], label='ML + Markowitz', alpha=0.7)
plt.title('Turnover hebdomadaire')
plt.ylabel('Somme des changements de poids')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nTurnover moyen ML Long-Only  : {turnover_ml_ts.iloc[1:].mean():.4f}")
print(f"Turnover moyen ML + Markowitz : {turnover_mkw_ts.iloc[1:].mean():.4f}")

## 9. Discussion et Conclusion

### Ce qui a bien fonctionné
- Le feature engineering avec indicateurs techniques, information sectorielle et corrélation au marché fournit des features pertinentes pour 50 actions.
- L'optimisation Markowitz avec prédictions ML comme rendements attendus et matrice de covariance historique permet de construire un portefeuille cohérent.
- Le split temporel strict et le rééquilibrage hebdomadaire sont réalistes.
- La contrainte de poids maximum (10% par action) limite la concentration.

### Ce qui n'a pas fonctionné
- Les R² sont très faibles (proches de 0 ou négatifs) : c'est normal en finance, le signal est très bruité.
- Le LSTM n'apporte pas de gain évident par rapport au Random Forest.
- Le turnover est élevé, ce qui en pratique génère des coûts de transaction.

### Limites
- Pas de prise en compte des coûts de transaction.
- La matrice de covariance sur 60 jours peut être instable.
- Pas de contraintes de liquidité.
- L'univers est fixe (pas de filtre dynamique sur la qualité des actions).

### Améliorations possibles
- Ajouter une pénalité de turnover dans l'optimisation.
- Tester la risk-parity comme alternative à Markowitz.
- Intégrer des données macro ou du sentiment via NLP.
- Utiliser XGBoost ou un Transformer pour les prédictions.
- Appliquer un filtre de liquidité dynamique.

### Conclusion
Ce projet montre qu'on peut utiliser le ML pour guider l'allocation d'un portefeuille de 50 actions NASDAQ. L'optimisation Markowitz couplée aux prédictions ML est plus sophistiquée que le simple equal weight, mais les rendements financiers restent extrêmement difficiles à prédire. Les gains dépendent fortement de la période testée.